# Bước 1: Thu Thập Dữ Liệu Lịch Sử VN100 (2022 - 2026)

**Môn học:** Học máy (Machine Learning) - HK7  
**Đề tài:** Phân nhóm cổ phiếu theo mức độ đồng biến động giá phục vụ đa dạng hóa danh mục đầu tư  

---

### Mục tiêu:
1. Xác định danh sách 100 cổ phiếu thuộc rổ **VN100** và phân ngành **ICB** tương ứng.
2. Tải dữ liệu giá lịch sử (OHLCV) theo ngày từ **01/01/2022 đến 23/10/2026** cho từng mã.
3. Lưu dữ liệu thô vào `data/raw/{symbol}.csv` và bảng thông tin mã vào `data/metadata/vn100_symbols.csv`.
4. Kiểm tra sơ bộ số lượng bản ghi của các file vừa tải.

In [ ]:
# Khai báo các thư viện cần thiết
import sys
from pathlib import Path
import pandas as pd

# Thiết lập đường dẫn thư mục gốc dự án
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_vn100_symbols, fetch_stock_history, save_symbol_data

print("Đã import thành công các module cần thiết!")

### 1. Lấy danh sách rổ cổ phiếu VN100 & Phân ngành ICB
Rổ VN100 gồm 30 mã VN30 (vốn hóa lớn) và 70 mã VNMidCap (vốn hóa trung bình, thanh khoản cao). Ta cần lưu thông tin nhóm ngành ICB để sau này đối chiếu với kết quả gom cụm của mô hình.

In [ ]:
# Lấy danh sách mã VN100
symbols, df_meta = get_vn100_symbols()
print(f"Tổng số mã cổ phiếu được nhận diện: {len(symbols)}")
print("10 mã đầu tiên:", symbols[:10])

# Lưu metadata vào data/metadata/vn100_symbols.csv
meta_path = PROJECT_ROOT / "data" / "metadata" / "vn100_symbols.csv"
meta_path.parent.mkdir(parents=True, exist_ok=True)
df_meta.to_csv(meta_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu metadata tại: {meta_path}")

### 2. Thu thập dữ liệu giá OHLCV từng mã (2022 - 2026)
Tải dữ liệu từ ngày `2022-01-01` đến `2026-10-23`. Mỗi mã được lưu thành một file CSV riêng trong `data/raw/` để tiện quản lý và tránh lỗi ngắt kết nối giữa chừng.

In [ ]:
import time
from tqdm import tqdm

start_date = "2022-01-01"
end_date = "2026-10-23"
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

success_list = []
fail_list = []

print(f"Bắt đầu cào dữ liệu {len(symbols)} mã từ {start_date} đến {end_date}...")
for sym in tqdm(symbols, desc="Thu thập giá VN100"):
    target_csv = raw_dir / f"{sym}.csv"
    if target_csv.exists():
        success_list.append(sym)
        continue
        
    df_price = fetch_stock_history(sym, start_date=start_date, end_date=end_date, source="AUTO")
    if df_price is not None and not df_price.empty:
        save_symbol_data(df_price, sym, raw_dir)
        success_list.append(sym)
    else:
        fail_list.append(sym)
    time.sleep(0.5)  # Tránh rate limit

print(f"\nHoàn tất: Thành công {len(success_list)}/{len(symbols)} mã.")
if fail_list:
    print("Các mã gặp sự cố:", fail_list)

### 3. Kiểm tra mẫu dữ liệu thô vừa thu thập
Đọc thử file của một mã tiêu biểu (ví dụ: HPG) để kiểm tra các cột dữ liệu trước khi chuyển sang bước tiền xử lý.

In [ ]:
sample_sym = "HPG"
sample_file = raw_dir / f"{sample_sym}.csv"
if sample_file.exists():
    df_sample = pd.read_csv(sample_file)
    print(f"=== Thông tin mẫu dữ liệu {sample_sym} ===")
    print(f"Số dòng: {len(df_sample)}")
    print("Các cột:", list(df_sample.columns))
    display(df_sample.head())
else:
    print(f"Chưa tìm thấy file {sample_file}")

### Tổng kết Bước 1 & Chuyển sang Bước 2:
Dữ liệu thô của 100 mã VN100 đã được tải và lưu đầy đủ tại `data/raw/`.  
Ở **Notebook 02**, ta sẽ:
1. Ghép 100 file thành một ma trận giá đóng cửa điều chỉnh (`adj_close`).
2. Lọc bỏ các mã thiếu dữ liệu (ngưỡng tối thiểu 90% số phiên).
3. Lọc bỏ các ngày nghỉ lễ sàn đóng cửa và điền khuyết các phiên thiếu cục bộ.